# 🎓 Mocogi MCP Client Demo

Dieses Notebook demonstriert die Nutzung des **Mocogi MCP Clients**, um Informationen über Studiengänge und Module der TH Köln abzufragen.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/modul_anerkennung/blob/master/notebooks/mcp_client_demo.ipynb)

## 🛠️ Installation und Setup

In [ ]:
!pip install git+https://github.com/dgaida/modul_anerkennung.git
!pip install fastmcp httpx llm-client gradio

## 🔑 Konfiguration

Hier kannst du deinen API-Key für das LLM und (optional) den Mocogi API Token setzen.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    # os.environ["MOCOGI_API_TOKEN"] = userdata.get("MOCOGI_API_TOKEN")
except Exception:
    print("Bitte stelle sicher, dass OPENAI_API_KEY in den Colab Secrets gesetzt ist.")
    os.environ["GROQ_API_KEY"] = "DEIN_API_KEY"

## 🖥️ Gradio Interface

Wir nutzen `llm_client` und verbinden uns mit dem MCP Server (hier simuliert durch direkten Import der Tools für die Demo-Zwecke im Notebook).

In [ ]:
import gradio as gr
import json
from llm_client import LLMClient
from modul_anerkennung.mcp_client import MocogiClient

# Initialisiere den LLM Client und den Mocogi MCP Client
# Hinweis: Stelle sicher, dass die entsprechenden API-Keys in der Umgebung gesetzt sind.
client = LLMClient(api_choice="gemini", llm="gemini-2.0-flash-exp", use_async=True)
mcp_client = MocogiClient()

async def ask_mcp(question):
    messages = [{"role": "user", "content": question}]
    
    async with mcp_client as mcp:
        # 1. Liste alle verfügbaren Tools des MCP Servers auf
        mcp_tools = await mcp.list_tools()
        
        # 2. Konvertiere MCP Tools in das Format für llm_client (OpenAI/Gemini Format)
        tools = []
        for tool in mcp_tools:
            tools.append({
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                }
            })
        
        # 3. LLM entscheidet, welches Tool aufgerufen werden soll
        # Wir führen maximal 5 Iterationen für Tool-Calls durch
        for _ in range(5):
            response = await client.achat_completion_with_tools(
                messages=messages,
                tools=tools
            )
            
            # Falls das LLM direkt antwortet ohne Tool-Call
            if not response.get("tool_calls"):
                return response.get("content", "")
            
            # Falls Tool-Calls vorhanden sind, führen wir sie aus
            messages.append({"role": "assistant", "content": response.get("content"), "tool_calls": response["tool_calls"]})
            
            for tool_call in response["tool_calls"]:
                tool_name = tool_call["function"]["name"]
                tool_args = json.loads(tool_call["function"]["arguments"])
                
                # Tool über den MCP Client aufrufen
                result = await mcp.call_tool(tool_name, tool_args)
                
                # Ergebnis zurück an den Chat-Verlauf übergeben
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call["id"],
                    "name": tool_name,
                    "content": json.dumps(result)
                })
        
    return "Maximale Anzahl an Tool-Calls erreicht."

iface = gr.Interface(
    fn=ask_mcp,
    inputs="text",
    outputs="text",
    title="Mocogi MCP Client Assistant",
    description="Frage nach Studiengängen oder Modulen der TH Köln. Das LLM nutzt nun autonom die MCP Tools."
)

iface.launch(debug=True, share=True)